# 01 - KDE mini repo ontology


## Goal

Scan the bundled mini KDE repo, run every per-format reader, and extract the ontology entities + relations. Learn what each file kind contributes to the knowledge graph.


## Prerequisites

- The kde_ontology_slm_lab repo cloned locally (or running on Colab after the bootstrap cell).
- Python 3.10+. No GPU. No model downloads.
- Familiarity with KDE Frameworks at a basic level (QObject, KConfig, D-Bus) helps but is not required.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. Scan the mini repo

The scanner walks the repo and classifies every file by extension or filename. It lives in `src/repo_ingest/scanner.py`. We do **not** parse anything yet — the scanner only labels files so the per-format readers below can be dispatched.


In [ ]:
from src.repo_ingest.scanner import scan
from src.common.paths import MINI_REPO

report = scan(MINI_REPO)
print(f"scanned {len(report.files)} files")
for f in report.files:
    print(f"  {f.kind:14s} {f.rel_path}")


## 2. Run the per-format readers

Each reader takes a path and returns a small dataclass. The readers live in `src/repo_ingest/*_reader.py`. We deliberately keep them small and pure so the ontology layer above stays testable.


In [ ]:
from src.repo_ingest.cmake_reader import read_cmake
from src.repo_ingest.cpp_reader import read_cpp
from src.repo_ingest.qml_reader import read_qml
from src.repo_ingest.dbus_reader import read_dbus
from src.repo_ingest.kconfig_reader import read_kconfig
from src.repo_ingest.desktop_file_reader import read_desktop
from src.repo_ingest.log_reader import read_log

# Sample one of each kind so you can see the shape of the reader output.
first = lambda kind: report.by_kind(kind)[0].path if report.by_kind(kind) else None

print("cmake :", read_cmake(first('cmake')))
print("cpp   :", read_cpp(first('cpp_header')))
print("qml   :", read_qml(first('qml')))
print("dbus  :", read_dbus(first('dbus')))
print("kcfg  :", read_kconfig(first('kconfig')))
print("desk  :", read_desktop(first('desktop')))
print("log   :", read_log(first('log')))


## 3. Build the ontology bundle

An `ExtractionBundle` is just two collections: `entities` (a dict by id) and `relations` (a list). Each `from_*` helper takes a reader result and appends to the bundle. See `src/ontology/extractor.py` and `src/ontology/schema.py` for the type and relation whitelists.


In [ ]:
from src.ontology.extractor import (
    ExtractionBundle, from_cmake, from_cpp, from_qml, from_dbus,
    from_kconfig, from_desktop, from_log,
)
from src.ontology.schema import Entity
from src.common.ids import make_id

bundle = ExtractionBundle()
repo_id = bundle.add_entity(Entity(
    id=make_id('Repository', MINI_REPO.name),
    type='Repository', name=MINI_REPO.name, source_path=str(MINI_REPO),
))

for sf in report.by_kind('cmake'):
    from_cmake(bundle, read_cmake(sf.path), repo_id)
for sf in report.by_kind('cpp_header') + report.by_kind('cpp_source'):
    from_cpp(bundle, read_cpp(sf.path))
for sf in report.by_kind('qml'):
    from_qml(bundle, read_qml(sf.path))
for sf in report.by_kind('dbus'):
    from_dbus(bundle, read_dbus(sf.path))
for sf in report.by_kind('kconfig'):
    from_kconfig(bundle, read_kconfig(sf.path))
for sf in report.by_kind('desktop'):
    from_desktop(bundle, read_desktop(sf.path))
for sf in report.by_kind('log'):
    from_log(bundle, read_log(sf.path))

print(f"entities : {len(bundle.entities)}")
print(f"relations: {len(bundle.relations)}")


## 4. Inspect the top entity types

A quick histogram tells you what the repo is *made of*. In a real KDE app you would expect `CppClass`, `Signal`, and `ConfigKey` to dominate.


In [ ]:
from collections import Counter

types = Counter(e.type for e in bundle.entities.values())
rels = Counter(r.rel for r in bundle.relations)

print('Top entity types:')
for t, n in types.most_common(10):
    print(f'  {n:3d}  {t}')
print()
print('Top relation types:')
for r, n in rels.most_common(10):
    print(f'  {n:3d}  {r}')


## 5. Spot-check one entity end-to-end

Walk one `CppClass` entity to confirm we have the right `name`, `source_path`, and `source_line`. These three fields are what every downstream answer cites.


In [ ]:
cpp_classes = [e for e in bundle.entities.values() if e.type == 'CppClass']
for c in cpp_classes:
    print(f"{c.name:30s} {c.source_path}:{c.source_line}")


## Summary

You now have a `bundle` containing every ontology entity and relation extracted from the mini repo. Notebook 02 turns this bundle into a NetworkX graph you can query, save, and visualise.


## Exercises

1. Add a new `.desktop` file to `examples/mini_kde_repo/desktop/` and re-run the cells. Confirm a new `DesktopFile` entity appears.
2. Change `src/repo_ingest/cpp_reader.py` to also capture `Q_INVOKABLE` methods. Verify the entity count grows.
3. Print every entity whose `source_path` is empty. What do they have in common, and is that intentional?
